In [1]:
from pathlib import Path
import pandas as pd

sample_path = Path("../data/sample/sample_jobs.csv")
jobs = pd.read_csv(sample_path, dtype={"job_id": str})

print("Number of jobs:", len(jobs))
print("Missing descriptions:", jobs["description"].isna().sum())

jobs[["job_id", "title", "location"]].head()

Number of jobs: 500
Missing descriptions: 0


,job_id,title,location
0,3901384985,Underwriter II / Senior Underwriter,"New York, United States"
1,3904954171,Lead Customer service Representative,"Hickory, NC"
2,3844654703,Account Manager,"Los Angeles, CA"
3,3904991666,Nurse Assistant - 6W Cardiac Stepdown - Full T...,"Mason City, IA"
4,3905324896,Psychology Technician - Department of Health P...,"Columbia, MO"


In [2]:
candidate = {
    "candidate_id": "synthetic_data_analyst",
    "skills": ["Python", "SQL", "Excel", "data analysis", "statistics"],
    "education": "Bachelor's degree in mathematics",
    "experience": (
        "Entry-level candidate with academic projects in data analysis. "
        "Used Python and SQL to clean data, analyze datasets, "
        "and communicate findings through reports and visualizations."
    ),
}

candidate_text = " ".join([
    " ".join(candidate["skills"]),
    candidate["education"],
    candidate["experience"],
])

print(candidate_text)

Python SQL Excel data analysis statistics Bachelor's degree in mathematics Entry-level candidate with academic projects in data analysis. Used Python and SQL to clean data, analyze datasets, and communicate findings through reports and visualizations.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Fit the vocabulary and word weights on job descriptions only.
vectorizer = TfidfVectorizer(stop_words="english")
job_vectors = vectorizer.fit_transform(jobs["description"])

# Represent the candidate using the same vocabulary and weights.
candidate_vector = vectorizer.transform([candidate_text])

# Compare this candidate with every job.
text_scores = cosine_similarity(candidate_vector, job_vectors).ravel()

results = jobs[["job_id", "title", "location"]].copy()
results["text_score"] = text_scores

top_10 = results.sort_values(
    ["text_score", "job_id"],
    ascending=[False, True],
).head(10)

top_10

,job_id,title,location,text_score
161,3902343369,Data Engineer,United States,0.271928
65,3894544709,Senior Data Analyst,"McLean, VA",0.243944
138,3903457662,Data Entry clerk.,United States,0.228296
262,3905302758,IT Data Engineer,"Atlanta, GA",0.187395
116,3901642507,Statistician,"Charlotte, NC",0.176310
476,3905215190,Azure Data Architect,"Roseville, CA",0.162160
206,3901941388,Sr. Data Engineer (onsite),"Bloomington, IN",0.161382
365,3894546475,Data Manager and Analytical Support Specialist...,"Bronx, NY",0.160081
6,3905256395,Sales Operations Specialist,"Columbia, MD",0.159647
250,3904958394,Technical Lead Information Intelligent Cloud S...,"Orlando, FL",0.158404


In [4]:
for job_id in top_10["job_id"].head(3):
    job = jobs.loc[jobs["job_id"] == job_id].iloc[0]

    print(job["title"])
    print(str(job["description"])[:2500])
    print("\n" + "=" * 80 + "\n")

Data Engineer
Work with teams and stakeholders across the business to understand data needs to design performant, scalable, and low-technical-debt solutions in a multi-cloud environment.Specialize in Azure data offerings like SQL MI, Redis Cache, and PostgreSQL.Utilize cloud-native services for data storage, processing, and analytics.Deploy data integration processes that are efficient and reliable and maximize data flow across the organization.Implement security protocols to safeguard sensitive data and enforce data governance.Boost data architectures for performance and scalability.Validate data quality and consistency throughout the data lifecycle.Build strong partnerships with product managers, developers, and other stakeholders to understand requirements and provide technical guidance.Key Qualifications:Bachelor's degree in computer science or similar degree preferred.3+ years of hands-on experience with cloud providers required.Proficient in SQL, PostGres, ElasticSearch, Redis an

In [5]:
candidates = [
    candidate,
    {
        "candidate_id": "synthetic_accountant",
        "skills": [
            "accounting", "Excel", "financial reporting",
            "bookkeeping", "account reconciliation",
        ],
        "education": "Bachelor's degree in accounting",
        "experience": (
            "Two years of accounting experience preparing financial "
            "reports, reconciling accounts, and processing invoices."
        ),
    },
    {
        "candidate_id": "synthetic_marketing_specialist",
        "skills": [
            "digital marketing", "SEO", "Google Analytics",
            "content creation", "social media",
        ],
        "education": "Bachelor's degree in marketing",
        "experience": (
            "Two years of marketing experience managing social media "
            "campaigns, writing content, and analyzing campaign performance."
        ),
    },
]

print([profile["candidate_id"] for profile in candidates])

['synthetic_data_analyst', 'synthetic_accountant', 'synthetic_marketing_specialist']


In [6]:
def profile_to_text(profile):
    """Combine declared candidate information into one text."""
    return " ".join([
        " ".join(profile["skills"]),
        profile["education"],
        profile["experience"],
    ])


rankings = {}

for profile in candidates:
    profile_vector = vectorizer.transform([profile_to_text(profile)])
    scores = cosine_similarity(profile_vector, job_vectors).ravel()

    ranked_jobs = jobs[["job_id", "title", "location"]].copy()
    ranked_jobs["text_score"] = scores

    rankings[profile["candidate_id"]] = ranked_jobs.sort_values(
        ["text_score", "job_id"],
        ascending=[False, True],
    ).head(10)


for candidate_id, ranking in rankings.items():
    print(candidate_id)
    display(ranking)

synthetic_data_analyst


,job_id,title,location,text_score
161,3902343369,Data Engineer,United States,0.271928
65,3894544709,Senior Data Analyst,"McLean, VA",0.243944
138,3903457662,Data Entry clerk.,United States,0.228296
262,3905302758,IT Data Engineer,"Atlanta, GA",0.187395
116,3901642507,Statistician,"Charlotte, NC",0.176310
476,3905215190,Azure Data Architect,"Roseville, CA",0.162160
206,3901941388,Sr. Data Engineer (onsite),"Bloomington, IN",0.161382
365,3894546475,Data Manager and Analytical Support Specialist...,"Bronx, NY",0.160081
6,3905256395,Sales Operations Specialist,"Columbia, MD",0.159647
250,3904958394,Technical Lead Information Intelligent Cloud S...,"Orlando, FL",0.158404


synthetic_accountant


,job_id,title,location,text_score
79,3904363876,Accounting Manager,"Oklahoma City, OK",0.501731
74,3901392467,Finance Coordinator,"Boston, MA",0.340185
370,3905883406,Senior Accountant,Topeka Metropolitan Area,0.317252
462,3902758248,Accounting Manager,"Salt Lake City, UT",0.274265
141,3905372329,"Associate Manager, Hedge Fund/Private Equity F...","Edina, MN",0.218016
28,3898161791,Staff Accountant,"Cedar Rapids, IA",0.203656
473,3901391232,Accounts Payable Specialist,"Detroit, MI",0.195322
307,3906256243,VP Corporate Financial Planning Analysis,"Pittsburgh, PA",0.183776
20,3900079355,Accounts Payable Analyst,"Kansas City, MO",0.174219
13,3900063970,Vice President Finance,"Pittsburgh, PA",0.173512


synthetic_marketing_specialist


,job_id,title,location,text_score
394,3894893823,Content Specialist,"Austin, TX",0.375379
214,3905669377,Public Affairs Specialist,"Washington, DC",0.322000
263,3904091394,Hiring- Web & Mobile Content Specialist- Merr...,"Merrifield, VA",0.281199
5,3895232336,Program Manager,"Charleston, South Carolina Metropolitan Area",0.268624
420,3891291458,Design Internship,"Chicago, IL",0.207588
352,3904430618,Marketing Specialist,"Fort Lauderdale, FL",0.203658
248,3905354157,Digital Marketing Client Account Manager,"Arizona, United States",0.193400
26,3905886911,Marketing Assistant,"Savannah, GA",0.192164
281,3894980127,Senior Account Executive – Public Relations (F...,"New York, NY",0.186911
198,3901652587,Vice President of Sales & Marketing - Southern...,"Richmond, VA",0.170515


In [7]:
import sys
from pathlib import Path
import importlib
import numpy as np

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import src.matching as matching
importlib.reload(matching)

matcher = matching.TextMatcher().fit(jobs)
module_scores = matcher.score(candidates[0])

np.testing.assert_allclose(module_scores, text_scores)
print("OK: module scores match the original notebook scores.")

OK: module scores match the original notebook scores.


In [8]:
import src.recommendation as recommendation
importlib.reload(recommendation)

for profile in candidates:
    print(profile["candidate_id"])

    top_jobs = recommendation.recommend_top_k(
        profile, jobs, k=10, matcher=matcher
    )

    display(top_jobs[["job_id", "title", "location", "text_score"]])

synthetic_data_analyst


,job_id,title,location,text_score
0,3902343369,Data Engineer,United States,0.271928
1,3894544709,Senior Data Analyst,"McLean, VA",0.243944
2,3903457662,Data Entry clerk.,United States,0.228296
3,3905302758,IT Data Engineer,"Atlanta, GA",0.187395
4,3901642507,Statistician,"Charlotte, NC",0.176310
5,3905215190,Azure Data Architect,"Roseville, CA",0.162160
6,3901941388,Sr. Data Engineer (onsite),"Bloomington, IN",0.161382
7,3894546475,Data Manager and Analytical Support Specialist...,"Bronx, NY",0.160081
8,3905256395,Sales Operations Specialist,"Columbia, MD",0.159647
9,3904958394,Technical Lead Information Intelligent Cloud S...,"Orlando, FL",0.158404


synthetic_accountant


,job_id,title,location,text_score
0,3904363876,Accounting Manager,"Oklahoma City, OK",0.501731
1,3901392467,Finance Coordinator,"Boston, MA",0.340185
2,3905883406,Senior Accountant,Topeka Metropolitan Area,0.317252
3,3902758248,Accounting Manager,"Salt Lake City, UT",0.274265
4,3905372329,"Associate Manager, Hedge Fund/Private Equity F...","Edina, MN",0.218016
5,3898161791,Staff Accountant,"Cedar Rapids, IA",0.203656
6,3901391232,Accounts Payable Specialist,"Detroit, MI",0.195322
7,3906256243,VP Corporate Financial Planning Analysis,"Pittsburgh, PA",0.183776
8,3900079355,Accounts Payable Analyst,"Kansas City, MO",0.174219
9,3900063970,Vice President Finance,"Pittsburgh, PA",0.173512


synthetic_marketing_specialist


,job_id,title,location,text_score
0,3894893823,Content Specialist,"Austin, TX",0.375379
1,3905669377,Public Affairs Specialist,"Washington, DC",0.322000
2,3904091394,Hiring- Web & Mobile Content Specialist- Merr...,"Merrifield, VA",0.281199
3,3895232336,Program Manager,"Charleston, South Carolina Metropolitan Area",0.268624
4,3891291458,Design Internship,"Chicago, IL",0.207588
5,3904430618,Marketing Specialist,"Fort Lauderdale, FL",0.203658
6,3905354157,Digital Marketing Client Account Manager,"Arizona, United States",0.193400
7,3905886911,Marketing Assistant,"Savannah, GA",0.192164
8,3894980127,Senior Account Executive – Public Relations (F...,"New York, NY",0.186911
9,3901652587,Vice President of Sales & Marketing - Southern...,"Richmond, VA",0.170515


In [9]:
importlib.reload(matching)

matching.skill_coverage(
    candidate_skills=["Python", "SQL", "Excel"],
    job_skills=["Python", "SQL", "AWS"],
)

{'skill_score': 0.6666666666666666,
 'matched_skills': ['python', 'sql'],
 'unverified_skills': ['aws']}

In [10]:
from src.feature_extraction import extract_skills

jobs["description"] = jobs["description"].fillna("")
jobs["skill_labels"] = jobs["description"].apply(extract_skills)

print(
    "Jobs with detected skills:",
    jobs["skill_labels"].map(bool).sum(),
    "out of",
    len(jobs),
)

jobs[["title", "skill_labels"]].head(10)

Jobs with detected skills: 306 out of 500


,title,skill_labels
0,Underwriter II / Senior Underwriter,"[Risk Management, Excel]"
1,Lead Customer service Representative,[]
2,Account Manager,[]
3,Nurse Assistant - 6W Cardiac Stepdown - Full T...,[Patient Care]
4,Psychology Technician - Department of Health P...,[]
5,Program Manager,"[Google Analytics, Excel]"
6,Sales Operations Specialist,"[Process Improvement, Excel, Salesforce, CRM, ..."
7,"Director, Facilities","[Vendor Management, Contract Management]"
8,Cosmic Bingo Host,[]
9,Associate,[]


In [11]:
importlib.reload(matching)
importlib.reload(recommendation)
matcher = matching.TextMatcher().fit(jobs)
demo_rankings = {}

for profile in candidates:
    demo_rankings[profile["candidate_id"]] = recommendation.score_jobs(
        profile, jobs, matcher=matcher,
        skills_column="skill_labels", text_weight=0.5,
    )
    print(profile["candidate_id"])
    for method in ["text", "skill", "hybrid"]:
        print(method)
        top_jobs = recommendation.recommend_top_k(
            profile, jobs, k=10, matcher=matcher, method=method,
            skills_column="skill_labels", text_weight=0.5,
        )
        display(top_jobs[[
            "job_id", "title", "text_score", "skill_score", "hybrid_score",
            "matched_skills", "unverified_skills", "hybrid_uses_skills",
        ]])

synthetic_data_analyst
text


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3902343369,Data Engineer,0.271928,0.500000,0.385964,"[python, sql]","[aws, java]",True
1,3894544709,Senior Data Analyst,0.243944,0.333333,0.288639,"[data analysis, python, sql]","[agile, aws, data visualization, r, six sigma,...",True
2,3903457662,Data Entry clerk.,0.228296,NaN,0.228296,[],[],False
3,3905302758,IT Data Engineer,0.187395,0.333333,0.260364,[python],"[agile, machine learning]",True
4,3901642507,Statistician,0.176310,0.714286,0.445298,"[data analysis, excel, python, sql, statistics]","[r, tableau]",True
5,3905215190,Azure Data Architect,0.162160,1.000000,0.581080,[sql],[],True
6,3901941388,Sr. Data Engineer (onsite),0.161382,0.250000,0.205691,[python],"[manufacturing, power bi, spark]",True
7,3894546475,Data Manager and Analytical Support Specialist...,0.160081,0.600000,0.380041,"[data analysis, excel, sql]","[power bi, strategic planning]",True
8,3905256395,Sales Operations Specialist,0.159647,0.200000,0.179823,[excel],"[crm, process improvement, salesforce, tableau]",True
9,3904958394,Technical Lead Information Intelligent Cloud S...,0.158404,0.333333,0.245869,[sql],"[data visualization, project management]",True


skill


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3884446482,Product Data Hub Developer,0.111768,1.0,0.555884,[sql],[],True
1,3887878425,On-Air Scheduling Specialist,0.007147,1.0,0.503574,[excel],[],True
2,3889420286,Veterans Affairs Specialist,0.017156,1.0,0.508578,[data analysis],[],True
3,3891003793,Accounts Payable Analyst,0.026893,1.0,0.513446,[excel],[],True
4,3891072686,Regional Sales Representative - MN,0.019671,1.0,0.509835,[excel],[],True
5,3894596608,SVB - Associate I Portfolio Management- Fund B...,0.039376,1.0,0.519688,[excel],[],True
6,3898170803,Quality Assurance Technician,0.025204,1.0,0.512602,[excel],[],True
7,3899536321,Database Administrator,0.112415,1.0,0.556207,[sql],[],True
8,3901162785,Billing Specialist,0.028966,1.0,0.514483,[excel],[],True
9,3901168071,Billing Specialist,0.028966,1.0,0.514483,[excel],[],True


hybrid


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3905215190,Azure Data Architect,0.162160,1.0,0.581080,[sql],[],True
1,3899536321,Database Administrator,0.112415,1.0,0.556207,[sql],[],True
2,3884446482,Product Data Hub Developer,0.111768,1.0,0.555884,[sql],[],True
3,3894596608,SVB - Associate I Portfolio Management- Fund B...,0.039376,1.0,0.519688,[excel],[],True
4,3902866333,Payment Resolution Specialist -I (Hospital Den...,0.037024,1.0,0.518512,[excel],[],True
5,3904724004,Strategic Finance Director,0.035684,1.0,0.517842,[excel],[],True
6,3904988510,Customer Care Receptionist,0.034132,1.0,0.517066,[excel],[],True
7,3901370331,Certified Medical Assistant - Sleep Center,0.033828,1.0,0.516914,[statistics],[],True
8,3901162785,Billing Specialist,0.028966,1.0,0.514483,[excel],[],True
9,3901168071,Billing Specialist,0.028966,1.0,0.514483,[excel],[],True


... (8 outputs left out)

skill


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3902365634,SEO Writer (Article writing internship),0.098751,1.000000,0.549375,[seo],[],True
1,3904091394,Hiring- Web & Mobile Content Specialist- Merr...,0.281199,0.666667,0.473933,"[google analytics, seo]",[javascript],True
2,3895232336,Program Manager,0.268624,0.500000,0.384312,[google analytics],[excel],True
3,3889114336,Wordpress Developer,0.021699,0.333333,0.177516,[seo],"[git, javascript]",True
4,3894893823,Content Specialist,0.375379,0.250000,0.312689,[seo],"[content marketing, lead generation, project m...",True
5,3815847830,Staff Software Engineer (AHT),0.020322,0.000000,0.010161,[],[agile],True
6,3862373833,Construction Estimator,0.007344,0.000000,0.003672,[],[budgeting],True
7,3884429864,Enterprise Account Executive - Pacific North W...,0.053649,0.000000,0.026824,[],"[crm, lead generation, salesforce]",True
8,3884432411,Fluorescent Penetrant Inspector,0.000456,0.000000,0.000228,[],[manufacturing],True
9,3884434211,Quality Assurance Engineer (323816),0.007839,0.000000,0.003919,[],"[agile, scrum, sql]",True


hybrid


,job_id,title,text_score,skill_score,hybrid_score,matched_skills,unverified_skills,hybrid_uses_skills
0,3902365634,SEO Writer (Article writing internship),0.098751,1.000000,0.549375,[seo],[],True
1,3904091394,Hiring- Web & Mobile Content Specialist- Merr...,0.281199,0.666667,0.473933,"[google analytics, seo]",[javascript],True
2,3895232336,Program Manager,0.268624,0.500000,0.384312,[google analytics],[excel],True
3,3905669377,Public Affairs Specialist,0.322000,NaN,0.322000,[],[],False
4,3894893823,Content Specialist,0.375379,0.250000,0.312689,[seo],"[content marketing, lead generation, project m...",True
5,3904430618,Marketing Specialist,0.203658,NaN,0.203658,[],[],False
6,3905354157,Digital Marketing Client Account Manager,0.193400,NaN,0.193400,[],[],False
7,3905886911,Marketing Assistant,0.192164,NaN,0.192164,[],[],False
8,3889114336,Wordpress Developer,0.021699,0.333333,0.177516,[seo],"[git, javascript]",True
9,3901937903,"Senior Director, Technical Marketing",0.139816,NaN,0.139816,[],[],False


## Baseline limitations

- Recommendations use 500 historical postings, not current openings.
- Candidate profiles and selected categories are synthetic.
- Skill labels come from the shared dictionary-based extractor in
  src.feature_extraction. Detection coverage is not extraction accuracy.
- Skill coverage applies only to detected labels. A score of 1 can result
  from one matching detected skill and does not establish full eligibility.
- Missing skill evidence produces an unknown skill score; hybrid falls
  back to text similarity.
- Experience and mandatory qualifications are not checked separately.
- Hybrid weights and category aggregation settings are demonstration
  choices, not validated optimal parameters.
- Category rankings are computed independently of the user's selection.
  Top-m averaging does not eliminate category-size effects.
- Sensitivity results show ranking changes, not recommendation accuracy.
- Human relevance evaluation is pending. Scores are not probabilities
  of interviews or offers.

## Integration checks

In [12]:
print("Sample:", sample_path)
print("Jobs:", len(jobs))
print("Jobs with skills:", jobs["skill_labels"].map(bool).sum())
print("Candidate profiles:", len(candidates))

print("\nExample extracted skills:")
print(jobs.loc[jobs["skill_labels"].map(bool), "skill_labels"].head())

print("\nRanking tables:", list(demo_rankings))

Sample: ..\data\sample\sample_jobs.csv
Jobs: 500
Jobs with skills: 306
Candidate profiles: 3

Example extracted skills:
0                             [Risk Management, Excel]
3                                       [Patient Care]
5                            [Google Analytics, Excel]
6    [Process Improvement, Excel, Salesforce, CRM, ...
7             [Vendor Management, Contract Management]
Name: skill_labels, dtype: object

Ranking tables: ['synthetic_data_analyst', 'synthetic_accountant', 'synthetic_marketing_specialist']


In [13]:
jobs[["title", "job_function"]].head(10)

,title,job_function
0,Underwriter II / Senior Underwriter,Analyst; Finance
1,Lead Customer service Representative,Other
2,Account Manager,Business Development; Sales
3,Nurse Assistant - 6W Cardiac Stepdown - Full T...,Health Care Provider
4,Psychology Technician - Department of Health P...,Health Care Provider
5,Program Manager,Information Technology
6,Sales Operations Specialist,Sales
7,"Director, Facilities",Management; Manufacturing
8,Cosmic Bingo Host,Art/Creative; Design; Information Technology
9,Associate,NaN


In [14]:
def split_job_functions(value):
    """Parse category labels stored as semicolon-separated text."""
    if pd.isna(value):
        return []

    return sorted({
        label.strip()
        for label in value.split(";")
        if label.strip()
    })


job_categories = jobs[["job_id", "job_function"]].copy()
job_categories["job_function"] = job_categories["job_function"].apply(
    split_job_functions
)

job_categories = (
    job_categories
    .explode("job_function")
    .dropna(subset=["job_function"])
    .drop_duplicates(["job_id", "job_function"])
    .rename(columns={"job_function": "category"})
    .reset_index(drop=True)
)

print("Jobs with categories:", job_categories["job_id"].nunique())
print("Distinct categories:", job_categories["category"].nunique())

job_categories.head(10)

Jobs with categories: 491
Distinct categories: 35


,job_id,category
0,3901384985,Analyst
1,3901384985,Finance
2,3904954171,Other
3,3844654703,Business Development
4,3844654703,Sales
5,3904991666,Health Care Provider
6,3905324896,Health Care Provider
7,3895232336,Information Technology
8,3905256395,Sales
9,3895527556,Management


In [15]:
m = 5

profile = candidates[0]
scored = demo_rankings[profile["candidate_id"]]

category_jobs = job_categories.merge(
    scored[["job_id", "text_score"]],
    on="job_id",
    how="left",
    validate="many_to_one",
)

category_sizes = category_jobs.groupby("category").size()

best_jobs = (
    category_jobs
    .sort_values(
        ["category", "text_score", "job_id"],
        ascending=[True, False, True],
    )
    .groupby("category", sort=False)
    .head(m)
)

category_ranking = (
    best_jobs.groupby("category")
    .agg(
        category_score=("text_score", "mean"),
        jobs_used=("job_id", "size"),
    )
    .join(category_sizes.rename("jobs_available"))
    .reset_index()
    .sort_values(
        ["category_score", "category"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

category_ranking.insert(0, "rank", range(1, len(category_ranking) + 1))

print(profile["candidate_id"])
category_ranking.head(10)

synthetic_data_analyst


,rank,category,category_score,jobs_used,jobs_available
0,1,Information Technology,0.208347,5,98
1,2,Engineering,0.136563,5,50
2,3,Research,0.099144,5,13
3,4,Administrative,0.096056,5,18
4,5,Analyst,0.095474,5,13
5,6,Sales,0.086611,5,91
6,7,Accounting/Auditing,0.070610,5,22
7,8,Finance,0.070438,5,41
8,9,Other,0.069685,5,48
9,10,Business Development,0.060584,5,51


In [16]:
# Illustrative user choice; not used to calculate category scores.
selected_category = "Analyst"

selected_result = category_ranking.loc[
    category_ranking["category"] == selected_category
]

print("Candidate:", profile["candidate_id"])
print("Selected category:", selected_category)

if selected_result.empty:
    print("The selected category is not represented in this sample.")
else:
    print("Position in independent ranking:", int(selected_result.iloc[0]["rank"]))

print("\nTop alternative categories:")
display(
    category_ranking.loc[
        category_ranking["category"] != selected_category
    ].head(3)
)

Candidate: synthetic_data_analyst
Selected category: Analyst
Position in independent ranking: 5

Top alternative categories:


,rank,category,category_score,jobs_used,jobs_available
0,1,Information Technology,0.208347,5,98
1,2,Engineering,0.136563,5,50
2,3,Research,0.099144,5,13


In [17]:
importlib.reload(recommendation)

module_categories = recommendation.rank_job_categories(
    demo_rankings[candidates[0]["candidate_id"]],
    job_categories,
    score_column="text_score",
    m=5,
)

pd.testing.assert_frame_equal(module_categories, category_ranking)
print("OK: category rankings match.")

OK: category rankings match.


In [18]:
category_rankings = {}

for candidate_profile in candidates:
    candidate_id = candidate_profile["candidate_id"]
    scored_jobs = demo_rankings[candidate_id]

    print("\nCandidate:", candidate_id)

    for method in ["text", "hybrid"]:
        ranking = recommendation.rank_job_categories(
            scored_jobs,
            job_categories,
            score_column=f"{method}_score",
            m=5,
        )

        category_rankings[(candidate_id, method)] = ranking

        print("Method:", method)
        display(ranking.head(5))


Candidate: synthetic_data_analyst
Method: text


,rank,category,category_score,jobs_used,jobs_available
0,1,Information Technology,0.208347,5,98
1,2,Engineering,0.136563,5,50
2,3,Research,0.099144,5,13
3,4,Administrative,0.096056,5,18
4,5,Analyst,0.095474,5,13


Method: hybrid


,rank,category,category_score,jobs_used,jobs_available
0,1,Information Technology,0.530413,5,98
1,2,Finance,0.515380,5,41
2,3,Sales,0.512979,5,91
3,4,Other,0.506947,5,48
4,5,Accounting/Auditing,0.416666,5,22



Candidate: synthetic_accountant
Method: text


,rank,category,category_score,jobs_used,jobs_available
0,1,Finance,0.330290,5,41
1,2,Accounting/Auditing,0.298445,5,22
2,3,Sales,0.173328,5,91
3,4,Strategy/Planning,0.133774,2,2
4,5,Administrative,0.096100,5,18


... (2 outputs left out)


Candidate: synthetic_marketing_specialist
Method: text


,rank,category,category_score,jobs_used,jobs_available
0,1,Marketing,0.224101,5,23
1,2,Information Technology,0.195017,5,98
2,3,Sales,0.166538,5,91
3,4,Engineering,0.115863,5,50
4,5,Other,0.107197,5,48


Method: hybrid


,rank,category,category_score,jobs_used,jobs_available
0,1,Information Technology,0.239571,5,98
1,2,Marketing,0.195972,5,23
2,3,Engineering,0.157944,5,50
3,4,Sales,0.150893,5,91
4,5,Other,0.091419,5,48


In [19]:
# Illustrative choices; these are not inputs to category ranking.
selected_categories = {
    "synthetic_data_analyst": "Analyst",
    "synthetic_accountant": "Accounting/Auditing",
    "synthetic_marketing_specialist": "Marketing",
}

comparison_rows = []

for candidate_id, selected_category in selected_categories.items():
    for method in ["text", "hybrid"]:
        ranking = category_rankings[(candidate_id, method)]
        selected = ranking.loc[
            ranking["category"] == selected_category
        ]
        alternatives = ranking.loc[
            ranking["category"] != selected_category
        ].head(3)

        comparison_rows.append({
            "candidate_id": candidate_id,
            "method": method,
            "selected_category": selected_category,
            "selected_rank": (
                int(selected.iloc[0]["rank"])
                if not selected.empty else None
            ),
            "top_category": ranking.iloc[0]["category"],
            "alternative_categories": alternatives["category"].tolist(),
        })

category_comparison = pd.DataFrame(comparison_rows)
category_comparison

,candidate_id,method,selected_category,selected_rank,top_category,alternative_categories
0,synthetic_data_analyst,text,Analyst,5,Information Technology,"[Information Technology, Engineering, Research]"
1,synthetic_data_analyst,hybrid,Analyst,8,Information Technology,"[Information Technology, Finance, Sales]"
2,synthetic_accountant,text,Accounting/Auditing,2,Finance,"[Finance, Sales, Strategy/Planning]"
3,synthetic_accountant,hybrid,Accounting/Auditing,3,Finance,"[Finance, Sales, Other]"
4,synthetic_marketing_specialist,text,Marketing,1,Marketing,"[Information Technology, Sales, Engineering]"
5,synthetic_marketing_specialist,hybrid,Marketing,2,Information Technology,"[Information Technology, Engineering, Sales]"


In [20]:
sensitivity_rows = []
sensitivity_rankings = {}

for candidate_profile in candidates:
    candidate_id = candidate_profile["candidate_id"]
    selected_category = selected_categories[candidate_id]

    for text_weight in [0.5, 0.75, 1.0]:
        scored = recommendation.score_jobs(
            candidate_profile,
            jobs,
            matcher=matcher,
            skills_column="skill_labels",
            text_weight=text_weight,
        )

        for m in [3, 5, 10]:
            ranking = recommendation.rank_job_categories(
                scored,
                job_categories,
                score_column="hybrid_score",
                m=m,
            )

            sensitivity_rankings[(candidate_id, text_weight, m)] = ranking

            selected = ranking.loc[
                ranking["category"] == selected_category
            ]

            sensitivity_rows.append({
                "candidate_id": candidate_id,
                "text_weight": text_weight,
                "m": m,
                "top_category": ranking.iloc[0]["category"],
                "selected_rank": (
                    int(selected.iloc[0]["rank"])
                    if not selected.empty else None
                ),
                "top_3": ranking["category"].head(3).tolist(),
            })

sensitivity_results = pd.DataFrame(sensitivity_rows)
display(sensitivity_results)

,candidate_id,text_weight,m,top_category,selected_rank,top_3
0,synthetic_data_analyst,0.50,3,Information Technology,7,"[Information Technology, Finance, Other]"
1,synthetic_data_analyst,0.50,5,Information Technology,8,"[Information Technology, Finance, Sales]"
2,synthetic_data_analyst,0.50,10,Information Technology,11,"[Information Technology, Finance, Sales]"
3,synthetic_data_analyst,0.75,3,Information Technology,6,"[Information Technology, Other, Finance]"
4,synthetic_data_analyst,0.75,5,Information Technology,6,"[Information Technology, Other, Finance]"
5,synthetic_data_analyst,0.75,10,Information Technology,10,"[Information Technology, Sales, Finance]"
6,synthetic_data_analyst,1.00,3,Information Technology,3,"[Information Technology, Engineering, Analyst]"
7,synthetic_data_analyst,1.00,5,Information Technology,5,"[Information Technology, Engineering, Research]"
8,synthetic_data_analyst,1.00,10,Information Technology,5,"[Information Technology, Engineering, Sales]"
9,synthetic_accountant,0.50,3,Accounting/Auditing,1,"[Accounting/Auditing, Finance, Sales]"


## Category sensitivity observations

We compared text weights of 0.5, 0.75, and 1.0 and top-m aggregation
with m = 3, 5, and 10 for three synthetic profiles.

- Information Technology ranked first for the analyst profile in all
  nine configurations, while Analyst moved between positions 3 and 11.
- The accountant's leading category varied between Finance and
  Accounting/Auditing.
- For the marketing profile, Information Technology led at text weight
  0.5, while Marketing led in all text-only configurations.

These observations describe ranking sensitivity, not recommendation
accuracy. No configuration was selected as optimal. The existing
0.5 text weight and m = 5 remain demonstration defaults.

Top-m averaging avoids summing scores across all jobs, but does not
eliminate category-size effects. Categories with fewer than m eligible
jobs use all available scored jobs.